# KDD: Selección, preprocesamiento y transformación

Se seleccionan columnas, se limpia, se transforma, y se guarda el dataset final en `data/final/`, particionado por `anio` y `mes`

### Imports

In [1]:
import os, socket
from pathlib import Path

import pandas as pd
import requests
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

pd.set_option("display.max_colwidth", 80)

### Utilidades

- Constantes

In [2]:
proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent
staging_dir = proyecto / "data" / "staging"
final_dir = proyecto / "data" / "final"
tc_path = proyecto / "data" / "cambios.csv"
anios = [2023, 2024, 2025]
out_dir = proyecto / "01_EDA_KDD" / "temp" / "a_transformaciones"
tab_dir, web_dir = out_dir / "tablas", out_dir / "web"
for d in (tab_dir, web_dir):
    d.mkdir(parents=True, exist_ok=True)

def guardar_tabla(df, nombre):
    sdf = spark.createDataFrame(df) if isinstance(df, pd.DataFrame) else df
    sdf.coalesce(1).write.mode("overwrite").parquet(str(tab_dir / nombre))
    sdf.toPandas().to_json(web_dir / f"{nombre}.json", orient="records", force_ascii=False, date_format="iso", indent=1)
    return len(df) if isinstance(df, pd.DataFrame) else sdf.count()

- Spark

In [3]:
spark_master = os.environ.get("spark_master", "local[*]")

builder = (SparkSession.builder
           .appName("transformaciones")
           .master(spark_master)
           .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
           .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "4g"))
           .config("spark.sql.shuffle.partitions", "24")
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.sql.session.timeZone", "America/Lima"))

if spark_master.startswith("spark://"):
    builder = (builder.config("spark.driver.host", socket.gethostname())
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

### Verificación del cluster

In [4]:
sc = spark.sparkContext
estado = sc._jsc.sc().getExecutorMemoryStatus()

print(f"Master        : {sc.master}")
print(f"Executors     : {estado.size() - 1}")
print(f"Cores totales : {sc.defaultParallelism}")
print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
print("UI del job    : http://localhost:4040")

Master        : spark://spark-master:7077
Executors     : 0
Cores totales : 2
Memoria driver: 2.0 GB
UI del job    : http://localhost:4040


### Proceso anticipado

| Etapa | Dónde |
|---|---|
| **Selección** | Seleccion de columnas |
| **Preprocesamiento** | Preprocesamiento y limpieza |
| **Transformación** | Transformación |
| **Minería** | Técnicas de similitud/ANN/flujos/reglas |

### Carga del dataset

Lee los parquets de `data\staging`

In [5]:
assert staging_dir.exists(), \
    f"No se encuentra {staging_dir}; corre EDA/a_entendimiento.ipynb"

base = spark.read.parquet(str(staging_dir)).cache()
N = base.count()
print(f"Dataset cargado: {N:,} filas x {len(base.columns)} columnas")

Dataset cargado: 236,229 filas x 18 columnas


---
### 1. Selección de columnas

Se queda sólo con las columnas útiles para las técnicas siguientes, descartando las casi vacías

In [6]:
cols_simples = [c for c, t in base.dtypes if not t.startswith("array")]
nulos = (base.select([
            (100 * F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1)) / N).alias(c)
            for c in cols_simples])
         .toPandas().T[0])

descartar = sorted(set([c for c in cols_simples if nulos[c] > 95] + ["metodo", "codigo_proceso"]))
sel = base.drop(*descartar)

print("Descartadas:", descartar)
print("Seleccionadas:", sel.columns)

Descartadas: ['codigo_proceso', 'estado', 'metodo']
Seleccionadas: ['ocid', 'fecha', 'entidad', 'region', 'descripcion', 'items_desc', 'metodo_detalle', 'categoria', 'monto_referencial', 'moneda', 'n_postores', 'monto_adjudicado', 'proveedor', 'unspsc', 'anio']


- Se descartan `codigo_proceso`, `estado` y `metodo` (`estado` 100% nula, `metodo` reemplazable por `metodo_detalle`) 

- Quedan 15 columnas para limpiar y transformar

### 2. Preprocesamiento y limpieza

#### a. Auxiliares

In [7]:
def normalizar_texto(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.upper(F.trim(F.col(c))))
    return df

def quitar_duplicados(df):
    w = Window.partitionBy("ocid").orderBy(F.col("fecha").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

def filtrar_anios(df, anios):
    return df.where(F.col("anio").isin(anios))

SERIES_TC = {"USD": "PN01234PM", "EUR": "PN01235PM", "BRL": "PN01237PM", "GBP": "PN01238PM", "CAD": "PN01244PM"}
MESES = {"Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4, "May": 5, "Jun": 6, "Jul": 7, "Ago": 8, "Sep": 9, "Set": 9, "Oct": 10, "Nov": 11, "Dic": 12}

def descargar_tipo_cambio(ruta, anios):
    # promedio mensual del BCRP (S/ por unidad de moneda); se descarga una sola vez
    filas = []
    for moneda, serie in SERIES_TC.items():
        url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{serie}/json/{min(anios)}-1/{max(anios)}-12"
        for periodo in requests.get(url, timeout=30).json()["periods"]:
            mes, anio = periodo["name"].split(".")
            filas.append((moneda, int(anio), MESES[mes], float(periodo["values"][0])))
    pd.DataFrame(filas, columns=["moneda", "anio", "_mes", "tc"]).to_csv(ruta, index=False)

def convertir_a_soles(df, tc, columnas):
    df = (df.withColumn("moneda", F.coalesce(F.col("moneda"), F.lit("PEN")))
            .withColumn("_mes", F.month("fecha"))
            .join(F.broadcast(tc), ["moneda", "anio", "_mes"], "left")
            .withColumn("tc", F.when(F.col("moneda") == "PEN", F.lit(1.0)).otherwise(F.col("tc"))))
    for c in columnas:
        df = df.withColumn(c, F.col(c) * F.col("tc"))
    return df.drop("_mes")

def limpiar_montos(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.when(F.col(c) > 0, F.col(c)))
    return df

def filtrar_con_descripcion(df):
    return df.where(F.length(F.trim("descripcion")) > 0)

#### b. Flujo

In [8]:
pasos = [("staging", sel.count())]

limpio = normalizar_texto(sel, ["entidad", "region", "proveedor", "metodo_detalle", "categoria"])

limpio = quitar_duplicados(limpio)
pasos.append(("sin duplicados", limpio.count()))

limpio = filtrar_anios(limpio, anios)
pasos.append((f"fecha en {min(anios)}-{max(anios)}", limpio.count()))

if not tc_path.exists():
    descargar_tipo_cambio(tc_path, anios)
tipo_cambio = spark.createDataFrame(pd.read_csv(tc_path))

limpio = convertir_a_soles(limpio, tipo_cambio, ["monto_referencial", "monto_adjudicado"])
assert limpio.where("tc IS NULL").count() == 0, "hay procesos sin tipo de cambio"
limpio = limpio.drop("tc")
pasos.append(("montos en soles", limpio.count()))

limpio = limpiar_montos(limpio, ["monto_referencial", "monto_adjudicado"])

limpio = filtrar_con_descripcion(limpio)
pasos.append(("con descripción", limpio.count()))

pasos_df = pd.DataFrame(pasos, columns=["paso", "filas"]).assign(perdidas=lambda d: d.filas.shift(1) - d.filas)
pasos_df

,paso,filas,perdidas
0,staging,236229,NaN
1,sin duplicados,236229,0.0
2,fecha en 2023-2025,235954,275.0
3,montos en soles,235954,0.0
4,con descripción,235954,0.0


#### c. Análisis

- 275 procesos (~0.1%) quedan fuera por fecha fuera de 2023-2025. 

- Los 4,789 procesos (~2.0%) que no están en PEN ya no se filtran: sus montos se convierten a soles con el tipo de cambio promedio mensual del BCRP. 

- El filtro de descripción vacía no pierde ninguna fila adicional.

- En total, el preprocesamiento deja 235,954 filas (99.9% de las 236,229 de staging).

#### d. Recuento

- **Columnas categóricas** (`entidad`, `region`, `proveedor`, `metodo_detalle`, `categoria`) en mayúsculas y sin espacios extra

- **Texto libre** (`descripcion`+`items_desc` → `texto`) en minúsculas y sin tildes

- **Duplicados:** si un `ocid` aparece más de una vez se queda el registro más reciente

- **Fechas:** sólo los años cargados (2023-2025)

- **Moneda:** montos en USD, EUR, GBP, CAD y BRL convertidos a soles (tipo de cambio promedio mensual del BCRP); se conserva la moneda original en `moneda`

- **Montos ≤ 0:** se ponen en nulo, pero el proceso se conserva para texto y frecuencias

### 3. Transformación

#### a. Auxiliares

In [9]:
CON_TILDE, SIN_TILDE = "áéíóúüñ", "aeiouun"

def sin_tildes(c):
    return F.translate(F.lower(c), CON_TILDE, SIN_TILDE)

def normalizar_texto_libre(df):
    # para similitud (Jaccard/MinHash/LSH) y vecinos aproximados (TF-IDF/ANN)
    return (df
            .withColumn("texto", F.concat_ws(" ", "descripcion", "items_desc"))
            .withColumn("texto", sin_tildes(F.col("texto")))
            .withColumn("texto", F.trim(F.regexp_replace(F.regexp_replace("texto", r"[^a-z0-9 ]", " "), r"\s+", " "))))

def tokenizar(df):
    # para vecinos aproximados (TF-IDF / ANN): palabras sin stopwords en español
    stop = [s.translate(str.maketrans(CON_TILDE, SIN_TILDE)) for s in StopWordsRemover.loadDefaultStopWords("spanish")]
    df = RegexTokenizer(inputCol="texto", outputCol="_tok", pattern=r"[^a-z]+", minTokenLength=3).transform(df)
    df = StopWordsRemover(inputCol="_tok", outputCol="_tok2", stopWords=stop).transform(df)
    return df.withColumn("tokens", F.array_distinct("_tok2")).drop("_tok", "_tok2")

def generar_shingles(df, k=5):
    # para similitud (Jaccard/MinHash/LSH): k-shingles de caracteres.
    return df.withColumn("shingles", F.when(F.length("texto") >= k,
            F.array_distinct(F.expr(f"transform(sequence(1, length(texto) - {k} + 1), i -> substring(texto, i, {k}))")))
            .otherwise(F.array("texto")))

def agregar_tramos_monto(df):
    # para minería de flujos (DGIM) y reglas de asociación (ítems de canasta)
    return (df
            .withColumn("monto_final", F.coalesce("monto_adjudicado", "monto_referencial"))
            .withColumn("tramo_monto",
                F.when(F.col("monto_final").isNull(), "0_sin_monto")
                 .when(F.col("monto_final") <    50_000, "1_<50k")
                 .when(F.col("monto_final") <   200_000, "2_50k-200k")
                 .when(F.col("monto_final") < 1_000_000, "3_200k-1M")
                 .when(F.col("monto_final") < 5_000_000, "4_1M-5M")
                 .otherwise("5_>5M"))
            .withColumn("postor_unico", (F.col("n_postores") == 1).cast("int"))
            .withColumn("rubro", F.array_distinct(F.transform("unspsc", lambda x: F.substring(x, 1, 2))))
            .withColumn("mes", F.month("fecha"))
            .withColumn("periodo", F.date_format("fecha", "yyyy-MM")))

def construir_canasta(df):
    # para reglas de asociación (Apriori/FP-Growth): cada proceso como transacción clave=valor
    item = lambda k, c: F.when(F.col(c).isNotNull(), F.concat(F.lit(f"{k}="), F.col(c).cast("string")))
    return df.withColumn("canasta", F.array_distinct(F.concat(
            F.array_compact(F.array(
                item("cat", "categoria"), item("met", "metodo_detalle"), item("reg", "region"),
                item("monto", "tramo_monto"),
                F.when(F.col("postor_unico") == 1, F.lit("comp=postor_unico"))
                 .when(F.col("n_postores") > 1, F.lit("comp=varios_postores")))),
            F.transform("rubro", lambda r: F.concat(F.lit("rubro="), r)))))

#### b. Flujo

In [10]:
t = normalizar_texto_libre(limpio)
t = tokenizar(t)
t = generar_shingles(t)
t.select("texto", "tokens", F.slice("shingles", 1, 6).alias("shingles (6 primeros)")).show(3, truncate=60)

t = agregar_tramos_monto(t)
t = construir_canasta(t)

postor_tramo = (t.groupBy("tramo_monto").agg(F.count("*").alias("procesos"),
                             F.round(100 * F.avg("postor_unico"), 1).alias("%_postor_unico")).orderBy("tramo_monto").cache())
postor_tramo.show()
t.select("canasta").show(3, truncate=110)

+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------+
|                                                       texto|                                                      tokens|                     shingles (6 primeros)|
+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------+
|servicio de manejo forestal de la reserva ecologica del r...|[servicio, manejo, forestal, reserva, ecologica, rio, rimac]|[servi, ervic, rvici, vicio, icio , cio d]|
|servicio de arrendamiento de inmueble para el juzgado de ...|[servicio, arrendamiento, inmueble, juzgado, paz, letrado...|[servi, ervic, rvici, vicio, icio , cio d]|
|contratacion del servicio de lavado y planchado de ropa h...|[contratacion, servicio, lavado, planchado, ropa, hospita...|[contr, ontra, ntrat, trata, ratac, ataci]

+-----------+--------+--------------+
|tramo_monto|procesos|%_postor_unico|
+-----------+--------+--------------+
|0_sin_monto|   10837|          19.3|
|     1_<50k|   15440|          43.8|
| 2_50k-200k|  104222|          10.2|
|  3_200k-1M|   70887|           9.3|
|    4_1M-5M|   24361|          10.0|
|      5_>5M|   10207|           9.6|
+-----------+--------+--------------+



+------------------------------------------------------------------------------------------------------+
|                                                                                               canasta|
+------------------------------------------------------------------------------------------------------+
|         [cat=SERVICES, met=CONCURSO PÚBLICO, reg=LIMA, monto=4_1M-5M, comp=varios_postores, rubro=77]|
|    [cat=SERVICES, met=CONTRATACIÓN DIRECTA, reg=SANTA, monto=2_50k-200k, comp=postor_unico, rubro=80]|
|[cat=SERVICES, met=ADJUDICACIÓN SIMPLIFICADA, reg=LIMA, monto=4_1M-5M, comp=varios_postores, rubro=76]|
+------------------------------------------------------------------------------------------------------+
only showing top 3 rows



#### c. Análisis

- `tokens` y `shingles` se ven limpios (minúsculas, sin tildes ni signos) y consistentes entre descripciones parecidas. 

- La tabla por `tramo_monto` confirma lo que ya se mostro. Cae de 43.8% en `1_<50k` a ~10% en los tramos de más de 200k — los procesos de montos chicos concentran mucha más baja competencia

- `canasta` queda lista como conjunto de ítems `clave=valor` (`cat=`, `met=`, `reg=`, `monto=`,
  `comp=`, `rubro=`): tamaño variable porque algunos de los 5 ítems base
  pueden faltar (si la columna de origen es nula) y `rubro=` aporta un ítem por cada código UNSPSC
  distinto (0, 1 o varios)

---
### 4. Transformaciones por técnica

#### 6.1 Similitud

- **Transformación:** `normalizar_texto_libre` + `generar_shingles` 

- **Columna resultante:** `shingles` — conjunto de k-shingles de 5 caracteres por proceso, listo para Jaccard/MinHash/LSH

#### 6.2 Vecinos aproximados

- **Transformación:** `normalizar_texto_libre` + `tokenizar`

- **Columna resultante:** `tokens` — palabras sin *stopwords*, lista para vectorizar con TF-IDF y usar en ANN (IVF/HNSW) junto a
`monto_final`

#### 6.3 Minería de flujos

- **Transformación:** `agregar_tramos_monto`, específicamente `postor_unico` y las columnas de tiempo (`fecha`, `mes`, `periodo`)

- **Columnas resultantes:** `postor_unico`, `fecha` — listas para DGIM (ventanas sobre el flujo) y para Bloom/Count-Min sobre `entidad`/`proveedor`

#### 6.4 Reglas de asociación

- **Transformación:** `agregar_tramos_monto` + `construir_canasta`

- **Columna resultante:** `canasta` — cada proceso como transacción de ítems `clave=valor`, lista
  para Apriori/FP-Growth. Ítems que puede contener:
  - `cat=<categoria>` (ej. `cat=GOODS`)
  - `met=<metodo_detalle>` (ej. `met=CONTRATACIÓN DIRECTA`)
  - `reg=<region>` (ej. `reg=LIMA`)
  - `monto=<tramo_monto>` (ej. `monto=1_<50k`)
  - `comp=postor_unico` o `comp=varios_postores` (ausente si `n_postores` es nulo)
  - `rubro=<XX>` — uno por cada código UNSPSC de 2 dígitos distinto del proceso (0, 1 o varios)

### 5. Dataset final y partición

Se guarda en `data/final/`, particionado por `anio` y `mes`

In [11]:
columnas_finales = ["ocid", "fecha", "anio", "mes", "periodo", "entidad", "region", 
                "proveedor", "categoria", "metodo_detalle", "n_postores", "postor_unico", 
                "moneda", "monto_referencial", "monto_adjudicado", "monto_final", "tramo_monto",
                "descripcion", "texto", "tokens", "shingles", "unspsc", "rubro", "canasta"]

final = t.select(*columnas_finales)
final.write.mode("overwrite").partitionBy("anio", "mes").parquet(str(final_dir))

procesos = spark.read.parquet(str(final_dir))
procesos.printSchema()

print(f"Filas finales: {procesos.count():,} | particiones en disco: " f"{len(list(final_dir.glob('anio=*/mes=*')))} carpetas anio=/mes=")
procesos.groupBy("anio").count().orderBy("anio").show()

root
 |-- ocid: string (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- periodo: string (nullable = true)
 |-- entidad: string (nullable = true)
 |-- region: string (nullable = true)
 |-- proveedor: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- metodo_detalle: string (nullable = true)
 |-- n_postores: integer (nullable = true)
 |-- postor_unico: integer (nullable = true)
 |-- moneda: string (nullable = true)
 |-- monto_referencial: double (nullable = true)
 |-- monto_adjudicado: double (nullable = true)
 |-- monto_final: double (nullable = true)
 |-- tramo_monto: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- texto: string (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- shingles: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- unspsc: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rubro: array (nullable =

Filas finales: 235,954 | particiones en disco: 36 carpetas anio=/mes=


+----+-----+
|anio|count|
+----+-----+
|2023|76390|
|2024|80263|
|2025|79301|
+----+-----+



#### a. Partición y plan de ejecución

- `data/final/` queda particionado físicamente en carpetas `anio=<año>/mes=<mes>/`

- Cuando un filtro usa esas columnas, Spark puede saltarse carpetas completas sin abrirlas (*partition pruning*)

- Cuando filtra por una columna que no es de partición, tiene que leer todas las carpetas

In [12]:
print("--- Filtro por columna de partición (anio, mes) ---")
procesos.where("anio = 2025 AND mes = 3").select("ocid").explain()

print("--- Filtro por columna NO particionada (categoria) ---")
procesos.where("categoria = 'bienes'").select("ocid").explain()

--- Filtro por columna de partición (anio, mes) ---
== Physical Plan ==
*(1) Project [ocid#6613]
+- *(1) ColumnarToRow
   +- FileScan parquet [ocid#6613,anio#6635,mes#6636] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final], PartitionFilters: [isnotnull(anio#6635), isnotnull(mes#6636), (anio#6635 = 2025), (mes#6636 = 3)], PushedFilters: [], ReadSchema: struct<ocid:string>


--- Filtro por columna NO particionada (categoria) ---
== Physical Plan ==
*(1) Project [ocid#6613]
+- *(1) Filter (isnotnull(categoria#6619) AND (categoria#6619 = bienes))
   +- *(1) ColumnarToRow
      +- FileScan parquet [ocid#6613,categoria#6619,anio#6635,mes#6636] Batched: true, DataFilters: [isnotnull(categoria#6619), (categoria#6619 = bienes)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final], PartitionFilters: [], PushedFilters: [IsNotNull(categoria), EqualTo(categoria,bienes)], ReadSchema: struct<oc

- Se confirma la diferencia: el primer plan muestra `PartitionFilters: [isnotnull(anio#...), isnotnull(mes#...), (anio#... = 2025), (mes#... = 3)]` — Spark descarta 35 de las 36 carpetas sin abrirlas

- El segundo plan (filtro por `categoria`) muestra `PartitionFilters: []` — tiene que escanear las 36 carpetas, y el filtro se aplica recién después de leer (`DataFilters`/`PushedFilters`)

### 6. Escritura de resultados

Se guardan las tablas en `01_EDA_KDD/temp/a_transformaciones/tablas/` (parquet) y `01_EDA_KDD/temp/a_transformaciones/web/` (json, para la página web)

In [13]:
tablas = {
    "seleccion_columnas": pd.DataFrame({"columna": nulos.index, "pct_nulo": nulos.values,
                                        "descartada": [c in descartar for c in nulos.index]}),
    "pasos_limpieza": pasos_df,
    "postor_unico_tramo": postor_tramo,
    "procesos_anio": procesos.groupBy("anio").count().orderBy("anio"),
}

escritos = {nombre: guardar_tabla(df, nombre) for nombre, df in tablas.items()}
print(f"Escrito en 01_EDA_KDD/temp/a_transformaciones/: {', '.join(escritos)}")

Escrito en 01_EDA_KDD/temp/a_transformaciones/: seleccion_columnas, pasos_limpieza, postor_unico_tramo, procesos_anio


#### a. Verificación

In [14]:
for nombre, n in escritos.items():
    leido = spark.read.parquet(str(tab_dir / nombre)).count()
    assert leido == n, f"{nombre}: esperaba {n}, leí {leido}"
    assert (web_dir / f"{nombre}.json").exists(), f"falta web/{nombre}.json"

print(f"OK: {len(escritos)} tablas (parquet + json)")

OK: 4 tablas (parquet + json)


---
### 7. Cerrar sesión

In [15]:
spark.stop()